<a id="curation-lab"></a>
# VideoDB Curation: Break a Dataset and See What Curation Catches

Curation checks a LeRobot v3 dataset against what its metadata declares, using the decoded video.
This notebook shows that on purpose-made problems. The **Curation Lab**, a small web app, changes a
dataset in Azure Blob **in place**: it shortens a video, shifts a camera, deletes a file. This
notebook audits the same files and puts what Curation reported next to what the Lab said it would.

* LeRobot only. The dataset is **connected**: Curation reads it where it lives, nothing is copied.
* Curation reads the Blob with a read-only SAS kept in your deployment's Key Vault. The notebook
  passes only the secret's **name**; the SAS never appears here.

**Flow:** connect while the dataset is clean (a dataset already missing a file is refused at
connect). Then apply a preset in the Lab and re-run from **What the Lab changed** (end of section 3).
Repeat for each problem.

<a href="https://colab.research.google.com/github/video-db/videodb-cookbook/blob/main/guides/curation/curation_lab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a id="install"></a>
## 1. Install and connect

Set `VIDEO_DB_BASE_URL` (in your environment or `.env`) to a deployment with Curation enabled, and
enter your API key when prompted.

In [ ]:
!pip install -q "videodb>=0.5.0" python-dotenv pandas requests

In [ ]:
import os
import time
from getpass import getpass

import pandas as pd
import requests
from dotenv import load_dotenv
from IPython.display import display
from videodb import connect
from videodb.exceptions import InvalidRequestError

load_dotenv()
api_key = os.getenv("VIDEO_DB_API_KEY") or getpass("Enter your VideoDB API key: ")
base_url = os.getenv("VIDEO_DB_BASE_URL") or input("VideoDB endpoint (VIDEO_DB_BASE_URL): ")

conn = connect(api_key=api_key, base_url=base_url)
print("Connected to", base_url)

<a id="dataset"></a>
## 2. Choose the dataset

In the Lab, open the dataset and click **Connect**: it gives you this cell filled in. Paste it over
the cell below. To use your own dataset without the Lab, fill in the four storage values and leave
the Lab values empty.

The secret named here must exist in your VideoDB deployment's Key Vault and hold a read-only SAS
for the container, for example one made with `az storage container generate-sas --permissions r`.

In [ ]:
# Where the dataset lives: one Blob container, one folder holding meta/info.json.
ACCOUNT_URL = "https://<storage-account>.blob.core.windows.net"
CONTAINER = "<container>"
ROOT = "<folder>"

# Name (not value) of the Key Vault secret holding a read-only SAS for that container.
SAS_SECRET_NAME = "<secret-name>"

# Optional: the Curation Lab that changes this dataset, to compare against what it expects.
LAB_URL = ""
LAB_DATASET = ""

In [ ]:
def error_detail(error):
    """The server's code, message and detail behind a failed call."""
    try:
        body = error.response.json()
    except Exception:
        return {"message": str(error)}
    return {k: body.get(k) for k in ("code", "message", "detail") if body.get(k) is not None}


def wait_for(job, timeout=3600, interval=5):
    """Poll an audit until it finishes. Returns (succeeded, data or error)."""
    deadline = time.monotonic() + timeout
    while True:
        body = conn.session.get(job["output_url"], timeout=30).json()
        if body.get("status") not in ("processing", "in progress"):
            break
        if time.monotonic() > deadline:
            raise TimeoutError(f"job {job['id']} still processing after {timeout}s")
        time.sleep(interval)
    response = body.get("response") or {}
    if not response.get("success"):
        return False, {"code": response.get("code"), "message": response.get("message")}
    return True, response.get("data", {})


def paged(path, key, page_size=500, **params):
    """Read every item from a paged endpoint."""
    items, offset = [], 0
    while True:
        page = conn.get(path=path, params={**params, "offset": offset, "limit": page_size})
        items += page[key]
        offset += page_size
        if offset >= page["total"]:
            return items


def issue_codes(result):
    """Every issue code in one check's result, including per-camera issues."""
    codes = [i["code"] for i in result.get("issues", [])]
    for per_camera in (result.get("sequences") or result.get("cameras") or {}).values():
        codes += [i["code"] for i in per_camera.get("issues", [])]
    return sorted(set(codes))


def detail_codes(detail):
    """Issue codes in a dataset_invalid error's detail."""
    found = detail.get("detail") or {}
    errors = found.get("errors", []) if isinstance(found, dict) else found
    return sorted({e["code"] for e in errors if isinstance(e, dict) and e.get("code")})


def explain(rows):
    """Each issue of every episode that did not pass, with its camera and message."""
    for row in rows:
        if row["status"] == "passed":
            continue
        print(f"Episode {row['episode']}: {row['status']}")
        for check, result in row["checks"].items():
            found = [(None, i) for i in result.get("issues", [])]
            for camera, per_camera in (result.get("sequences") or result.get("cameras") or {}).items():
                found += [(camera, i) for i in per_camera.get("issues", [])]
            for camera, i in found:
                where = f" [{camera}]" if camera else ""
                print(f"  {check}{where}: {i['code']} ({i['status']}): {i['message']}")

<a id="connect"></a>
## 3. Connect the Blob dataset

`credential.kind = key_vault_secret` tells Curation to read the SAS from its Key Vault by name and
append it to every request. Connecting validates the dataset, so a missing or wrong secret, or a file
the metadata names but the Blob does not have, is refused here with its codes.

In [ ]:
dataset_id, connect_codes = None, []
try:
    dataset = conn.post(path="dataset", data={
        "name": "Curation Lab: SO-101 pick and place",
        "source": {
            "url": f"{ACCOUNT_URL}/{CONTAINER}",
            "root": ROOT,
            "credential": {"kind": "key_vault_secret", "name": SAS_SECRET_NAME},
        },
        "format": {"layout": "lerobot"},
    })
    dataset_id = dataset["id"]
    print("Dataset:", dataset_id, "| layout:", dataset["layout"], "| managed:", dataset["managed"])
    display(dataset["summary"])
except InvalidRequestError as error:
    detail = error_detail(error)
    connect_codes = detail_codes(detail)
    print("Refused:", detail.get("code"), "-", detail.get("message"))
    print("Codes:", connect_codes)
    print(detail.get("detail"))

### What the Lab changed

Reads the Lab's change log: each change still in effect, and what Curation should report for it.
Skipped when `LAB_URL` is empty. Re-run it after each change you make in the Lab.

In [ ]:
expected = []
if LAB_URL:
    if "lab" not in globals():
        lab = requests.Session()
        access_code = os.getenv("CURATION_LAB_ACCESS_CODE") or getpass("Curation Lab access code: ")
        lab.post(f"{LAB_URL}/api/session", json={"code": access_code}, timeout=30).raise_for_status()
    changes = lab.get(f"{LAB_URL}/api/datasets/{LAB_DATASET}/changes", timeout=30).json()["items"]
    active = [c for c in changes if c["state"] == "active"]
    for c in active:
        print(f"- {c['title']}: {c['summary']}")
        for e in c["expect"]:
            expected.append({"change": c["title"], **e})
    if not active:
        print("No change in effect: the dataset matches its pristine copy, so everything should pass.")
    display(pd.DataFrame(expected))
else:
    print("No Lab set: the results below are shown without expectations.")

<a id="validate"></a>
## 4. Validate

Validate checks the dataset's shape: its metadata, and that every file it names is there. A broken
reference (for example a deleted video) is reported here as an error with codes.

In [ ]:
validate_codes = connect_codes
if dataset_id is None:
    print("Not connected; connecting already reported:", connect_codes)
else:
    try:
        result = conn.post(path=f"dataset/{dataset_id}/validate")
        print("Valid:", result["valid"])
    except InvalidRequestError as error:
        detail = error_detail(error)
        validate_codes = detail_codes(detail)
        print("Invalid:", detail.get("message"), validate_codes)
        print(detail.get("detail"))

<a id="audit"></a>
## 5. Audit

The audit opens each episode's data and video and runs the two checks:

* **completeness**: every step has its frame, and the timelines agree;
* **camera_validity**: every camera has the declared size, frame rate and length.

By default it audits the episodes the Lab touched plus episode 0 as an untouched control; set
`EPISODES = "all"` to audit all 50.

In [ ]:
touched = sorted({n for e in expected if isinstance(e.get("episodes"), list) for n in e["episodes"]})
EPISODES = sorted(set(touched) | {0}) if touched else "all"
print("Auditing:", EPISODES)

succeeded, audit = False, {"code": None, "message": "not connected"}
if dataset_id is None:
    print("Skipped: the dataset is not connected.")
else:
    audit_job = conn.post(path=f"dataset/{dataset_id}/audit",
                          data={"episodes": EPISODES, "checks": ["completeness", "camera_validity"]})
    succeeded, audit = wait_for(audit_job)
    if succeeded:
        print("Overall status:", audit["status"])
        display(pd.DataFrame(audit["summary"]).T)
    else:
        print("The audit failed as a whole:", audit)

In [ ]:
rows = paged(f"dataset/{dataset_id}/audit/{audit_job['id']}/episodes", "episodes") if succeeded else []
table = pd.DataFrame([
    {"episode": r["episode"], "status": r["status"],
     **{check: result["status"] for check, result in r["checks"].items()},
     "issues": ", ".join(sorted({c for result in r["checks"].values() for c in issue_codes(result)}))}
    for r in rows
])
display(table)
explain(rows)

<a id="compare"></a>
## 6. Did Curation report what the Lab expected?

One row per expectation: the check, the episode, what was expected and what came back.

In [ ]:
by_episode = {r["episode"]: r["checks"] for r in rows}
verdicts = []
for e in expected:
    want = set(e["codes"])
    if e["check"] == "validate":
        verdicts.append({"change": e["change"], "check": "validate", "episode": "-",
                         "expected": f"{e['status']} {sorted(want)}", "got": validate_codes,
                         "match": want <= set(validate_codes)})
        continue
    if e["check"] == "audit":
        got = None if succeeded else audit.get("code")
        verdicts.append({"change": e["change"], "check": "audit", "episode": "-",
                         "expected": f"{e['status']} {sorted(want)}", "got": got or "completed",
                         "match": (not succeeded) and got in want,
                         **({"note": "not connected, so no audit ran"} if dataset_id is None else {})})
        continue
    episodes = e["episodes"] if isinstance(e["episodes"], list) else sorted(by_episode)
    for n in episodes:
        result = by_episode.get(n, {}).get(e["check"], {})
        got = (result.get("status"), issue_codes(result)) if result else None
        verdicts.append({"change": e["change"], "check": e["check"], "episode": n,
                         "expected": f"{e['status']} {sorted(want)}", "got": got,
                         "match": bool(result) and result.get("status") == e["status"] and want <= set(issue_codes(result))})

if verdicts:
    report = pd.DataFrame(verdicts)
    display(report)
    print("All as expected" if report["match"].all() else "Some results differ from the Lab's expectation")
elif succeeded:
    print("No expectations to compare. Statuses:", table["status"].value_counts().to_dict() if len(table) else {})

<a id="cleanup"></a>
## 7. Clean up

Forgets the connection. A connected dataset's files are never touched, so the Blob stays as the Lab left it.
Reset it from the Lab when you are done.

In [ ]:
if dataset_id:
    conn.delete(path=f"dataset/{dataset_id}")
    print("Disconnected", dataset_id)